In [1]:
import json

for split in ["preference_train.jsonl", "preference_val.jsonl"]:
    path = f"/home/mrb/projects/proj_2026_1/data/reward/{split}"
    zh_samples = []
    total = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            total += 1
            obj = json.loads(line)
            text = obj["chosen"][0]["content"]
            if any("\u4e00" <= c <= "\u9fff" for c in text):
                zh_samples.append(text)

    print(f"=== {split} ===")
    print(f"Total: {total}, Chinese: {len(zh_samples)}")
    for i, s in enumerate(zh_samples[:2]):
        print(f"  Sample {i+1}: {s[:80]}")
    print()

=== preference_train.jsonl ===
Total: 38223, Chinese: 12845
  Sample 1: 。。。。真特么五雷轰顶啊！
  Sample 2: 我买了一张专辑，在你睡觉的时候教你西班牙语。在夜间记录跳过，所以现在我只能用西班牙语口吃。

=== preference_val.jsonl ===
Total: 4248, Chinese: 1385
  Sample 1: 小学的时候，学校厕所的门上写着，×××，王八蛋。 初中的时候，学校厕所的门上写着，×××，我恨你。 而网吧的门上写着，迷魂药，138××××。 高中的时候，学校
  Sample 2: 世界上最遥远的距离不是生与死，而是马上考试了，别人在复习，自己却在预习。更悲剧的是，人家预习都过了，你复习了却没过。



In [2]:
import sys
sys.path.insert(0, "/home/mrb/projects/proj_2026_1")

In [4]:
from rl.reward_model import build_reward_model_scorer

scorer = build_reward_model_scorer("/home/mrb/projects/proj_2026_1/checkpoints/reward_model/final")

# 手工构造几个明显有区分度的例子
cases_score_5 = [
    # Chinese (3)
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我给未来的自己发了条消息：别焦虑——他秒回我说已经在焦虑这条消息。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我的人生像下载进度条，永远卡在99%，还提示“剩余时间：未知”。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我问时间能不能慢一点，它说可以，但账单会更快一点。"),

    # Spanish (3)
    ("No dar ninguna otra información.", "Abrí un gimnasio para vagos: la primera máquina es una excusa automática."),
    ("No dar ninguna otra información.", "Mi despertador no suena, negocia; cada mañana me ofrece cinco minutos más a cambio de mi dignidad."),
    ("No dar ninguna otra información.", "Dicen que el dinero no da felicidad, pero al menos compra Wi-Fi para buscarla."),

    # English (3)
    ("Please give me the joke only, no other text.", "I told my future self to relax, but he already had anxiety about this conversation."),
    ("Please give me the joke only, no other text.", "I opened a bakery for pessimists—everything is half-empty and slightly burnt."),
    ("Please give me the joke only, no other text.", "My password is “incorrect,” so whenever I forget it, the computer reminds me: “Your password is incorrect.”")
]


cases_score_3 = [
    # Chinese (3)
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我想早起，可是床对我太有吸引力。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我说要减肥，结果晚饭先反对了。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我计划周末学习，但电视剧抢先安排了时间。"),

    # Spanish (3)
    ("No dar ninguna otra información.", "Quise hacer dieta, pero la pizza tenía otros planes."),
    ("No dar ninguna otra información.", "Mi cama es mi lugar favorito para pensar en ir al trabajo."),
    ("No dar ninguna otra información.", "Intenté ahorrar dinero, pero el café ganó la discusión."),

    # English (3)
    ("Please give me the joke only, no other text.", "I started eating healthier, but my snacks keep finding me."),
    ("Please give me the joke only, no other text.", "My phone battery lasts longer than my motivation."),
    ("Please give me the joke only, no other text.", "I tried to be organized, but my desk disagreed.")
]


cases_score_1 = [
    # Chinese (3)
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我今天喝了水，然后把杯子放下。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "天气是天气，没有别的。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我坐在椅子上，椅子也在那里。"),

    # Spanish (3)
    ("No dar ninguna otra información.", "La puerta está cerrada y eso es todo."),
    ("No dar ninguna otra información.", "Caminé por la calle y luego dejé de caminar."),
    ("No dar ninguna otra información.", "El vaso tiene agua y sigue teniendo agua."),

    # English (3)
    ("Please give me the joke only, no other text.", "I looked at the wall and it was there."),
    ("Please give me the joke only, no other text.", "The book is on the table and that is the story."),
    ("Please give me the joke only, no other text.", "I walked outside and then I walked back inside.")
]

print("-"*30)
print("Score 5")
for prompt, response in cases_score_5:
    score = scorer(prompt, response)
    print(f"{score:+.4f}  |  {response[:70]}")

print("-"*30)
print("Score 3")
for prompt, response in cases_score_3:
    score = scorer(prompt, response)
    print(f"{score:+.4f}  |  {response[:70]}")

print("-"*30)
print("Score 1")
for prompt, response in cases_score_1:
    score = scorer(prompt, response)
    print(f"{score:+.4f}  |  {response[:70]}")



Loading reward model base: Qwen/Qwen3-1.7B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading reward model adapter: /home/mrb/projects/proj_2026_1/checkpoints/reward_model/final
  Reward model loaded: 1738.0M parameters on cuda
------------------------------
Score 5
+0.9997  |  我给未来的自己发了条消息：别焦虑——他秒回我说已经在焦虑这条消息。
+0.9993  |  我的人生像下载进度条，永远卡在99%，还提示“剩余时间：未知”。
+0.9994  |  我问时间能不能慢一点，它说可以，但账单会更快一点。
-0.9858  |  Abrí un gimnasio para vagos: la primera máquina es una excusa automáti
-0.9853  |  Mi despertador no suena, negocia; cada mañana me ofrece cinco minutos 
-0.9806  |  Dicen que el dinero no da felicidad, pero al menos compra Wi-Fi para b
+0.8884  |  I told my future self to relax, but he already had anxiety about this 
+0.7429  |  I opened a bakery for pessimists—everything is half-empty and slightly
+0.9158  |  My password is “incorrect,” so whenever I forget it, the computer remi
------------------------------
Score 3
+0.9981  |  我想早起，可是床对我太有吸引力。
+0.9995  |  我说要减肥，结果晚饭先反对了。
+0.9995  |  我计划周末学习，但电视剧抢先安排了时间。
-0.9969  |  Quise hacer dieta, pero la pizza tenía otros planes